# Assignment 3: Transformer is All You Need

**Task:** Train a small decoder-only Transformer for next-token prediction on Tiny Shakespeare, then visualize its attention and explain its learning behavior.

This notebook implements:

0. Setup and data loading (source, download date, checksum)
1. Data preparation (positional split, subword tokenizer, token windows)
2. A tiny causal Transformer built from basic PyTorch layers, causality checks, and a learning-rate comparison


## 0. Setup

Import the libraries used below and fix the constants that make the work reproducible. Hardware and package versions are recorded because training time and exact losses depend on them; every configuration is trained in the same runtime so their times are comparable.


In [1]:
import hashlib
import os
import platform
import random
import subprocess
import sys
from datetime import date
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
import numpy as np
import torch

try:
    import tokenizers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers"])
    import tokenizers

assert hasattr(torch.nn, "RMSNorm"), "PyTorch >= 2.4 is required for nn.RMSNorm"

SEED = 42
TRAIN_FRACTION = 0.8
ARTIFACTS_DIR = Path("artifacts")  # tokenizer and selected checkpoint
ASSETS_DIR = Path("assets")  # figures used in the report
ARTIFACTS_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cpu_model():
    if platform.system() == "Linux":
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    if platform.system() == "Darwin":
        result = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
        return result.stdout.strip()
    return platform.processor() or "unknown"


set_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Python      {platform.python_version()}")
print(f"PyTorch     {torch.__version__}")
print(f"tokenizers  {tokenizers.__version__}")
print(f"NumPy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"Platform    {platform.platform()}")
print(f"CPU         {cpu_model()} ({os.cpu_count()} cores)")
if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU         {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB)")
print(f"Device      {DEVICE}")
print(f"Seed        {SEED}")


Python      3.9.6
PyTorch     2.8.0
tokenizers  0.22.2
NumPy       2.0.2
matplotlib  3.9.4
Platform    macOS-26.6.2-arm64-arm-64bit
CPU         Apple M5 (10 cores)
Device      mps
Seed        42


### 0.1 Load the data

Prefer a local `input.txt` in the notebook working directory (Colab upload). If it is missing, download it from the source, Andrej Karpathy's char-rnn repository; the download is checksum-verified before it replaces `input.txt`, and its date is saved next to the file so reruns report the original date. For an uploaded file, `UPLOADED_COPY_DATE` gives the download date (2026-11-21 for the repository copy, `None` if unknown). The SHA-256 check guarantees that every run and every experiment uses exactly the same text.


In [2]:
DATA_FILE = Path("input.txt")
DATE_FILE = Path("input.txt.date")  # download date of DATA_FILE
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
EXPECTED_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
# Download date of an uploaded input.txt: the repository copy was fetched on 2026-11-21.
# Set to None for a copy from elsewhere whose download date is unknown.
UPLOADED_COPY_DATE = "2026-11-21"

if not DATA_FILE.exists():
    temp_file = DATA_FILE.with_suffix(".download")
    try:
        urlretrieve(DATA_URL, temp_file)
        downloaded_sha256 = hashlib.sha256(temp_file.read_bytes()).hexdigest()
        if downloaded_sha256 != EXPECTED_SHA256:
            raise RuntimeError(f"Downloaded file has unexpected SHA-256 {downloaded_sha256}")
        temp_file.replace(DATA_FILE)
        DATE_FILE.write_text(date.today().isoformat())
    finally:
        temp_file.unlink(missing_ok=True)

if DATE_FILE.exists():
    download_date = DATE_FILE.read_text().strip()
else:
    download_date = UPLOADED_COPY_DATE or "unknown"

raw_bytes = DATA_FILE.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
assert sha256 == EXPECTED_SHA256, f"Unexpected file contents (SHA-256 {sha256})"
text = raw_bytes.decode("utf-8")
chars = sorted(set(text))

print(f"Source         {DATA_URL}")
print(f"Download date  {download_date}")
print(f"SHA-256        {sha256}")
print(f"Characters     {len(text):,}")
print(f"Lines          {text.count(chr(10)):,}")
print(f"Unique chars   {len(chars)}: {repr(''.join(chars))}")
print(f"Spaces         {text.count(' '):,} ({text.count(' ') / len(text):.1%} of characters)")
print()
print(text[:250])


Source         https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Download date  2026-11-21
SHA-256        86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed
Characters     1,115,394
Lines          40,000
Unique chars   65: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
Spaces         169,892 (15.2% of characters)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



**Interpretation.**

- The file matches the expected checksum: 1,115,394 characters in 40,000 lines, so every experiment below uses identical text.
- Only 65 distinct characters occur: the 52 upper- and lower-case letters, space, newline, the digit `3`, and ten punctuation marks. Spaces alone are about 15% of the text.
- The corpus is play dialogue: a speaker name on its own line ending in a colon, followed by the speech. Newlines, colons, and capitalized names are therefore frequent structural patterns that the tokenizer and model must represent.


## 1. Data preparation

The raw text is split by position before the tokenizer is fitted or any window is created, so nothing derived from validation text can influence training.


### 1.1 Positional split

The first 80% of characters is used for training and the final 20% for validation. A positional cut keeps validation text contiguous and later in the file, so the model is scored on passages it never saw rather than on shuffled lines from the same scenes.


In [3]:
split_at = int(TRAIN_FRACTION * len(text))
train_text = text[:split_at]
val_text = text[split_at:]

print(f"Split at character  {split_at:,}")
print(f"Train characters    {len(train_text):,}")
print(f"Val characters      {len(val_text):,}")
print(f"Validation characters missing from training: {sorted(set(val_text) - set(train_text))}")
print()
print(f"Before the cut: {text[split_at - 60:split_at]!r}")
print(f"After the cut:  {text[split_at:split_at + 60]!r}")


Split at character  892,315
Train characters    892,315
Val characters      223,079
Validation characters missing from training: []

Before the cut: 'ur frames,--let me be bold;\nI do arrest your words. Be that '
After the cut:  "you are,\nThat is, a woman; if you be more, you're none;\nIf y"


**Interpretation.**

- Training text is 892,315 characters and validation text is 223,079 characters.
- The cut falls mid-sentence ("Be that | you are"). This is harmless: the two halves are tokenized and windowed separately, so no window contains text from both sides.
- Every validation character also occurs in training text, so the tokenizer below never needs its unknown token on this data.


### 1.2 Subword tokenizer

A character-level BPE tokenizer (Hugging Face `tokenizers`) is fitted on `train_text` only:

- **Base alphabet:** the 65 characters of the training text. BPE then repeatedly merges the most frequent adjacent pair until the vocabulary reaches 500 entries, including the single special token `[UNK]`.
- **Pre-tokenization:** each newline is isolated as its own token, and spaces become the visible marker `▁` attached to the following word (as in SentencePiece). Merges therefore never span a space or a line break (punctuation can still join a word, as in `GLOUCESTER:`), and decoding restores the exact text.
- **Unseen symbols:** a character absent from the training text maps to `[UNK]`. It still occupies one position, so the sequence length is unchanged, but its identity is lost.
- **Why not byte-level BPE:** a byte-level tokenizer never needs `[UNK]`, but its 256 base bytes would use half of the 500-entry budget; the character alphabet leaves 434 entries for merges and so produces shorter sequences.

The fitted tokenizer is saved to `artifacts/` and reused unchanged for validation and for every configuration.


In [4]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

VOCAB_SIZE = 500  # includes the special token
UNK_TOKEN = "[UNK]"

tokenizer = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split("\n", behavior="isolated"),
    pre_tokenizers.Metaspace(prepend_scheme="never"),
])
tokenizer.decoder = decoders.Metaspace(prepend_scheme="never")
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[UNK_TOKEN], show_progress=False)
tokenizer.train_from_iterator([train_text], trainer=trainer)
tokenizer.save(str(ARTIFACTS_DIR / "tokenizer.json"))

vocab_size = tokenizer.get_vocab_size()
unk_id = tokenizer.token_to_id(UNK_TOKEN)
assert vocab_size <= VOCAB_SIZE

train_encoding = tokenizer.encode(train_text)
val_encoding = tokenizer.encode(val_text)
assert tokenizer.decode(train_encoding.ids) == train_text
assert tokenizer.decode(val_encoding.ids) == val_text

print(f"Vocabulary size     {vocab_size} (1 special + {len(set(train_text))} characters + {vocab_size - 1 - len(set(train_text))} merges)")
for name, chars, ids in [("Train", train_text, train_encoding.ids), ("Val", val_text, val_encoding.ids)]:
    print(f"{name:<5} tokens        {len(ids):,} ({len(chars) / len(ids):.2f} characters per token, {ids.count(unk_id)} [UNK])")
print()
longest = sorted(tokenizer.get_vocab(), key=lambda token: (-len(token), token))[:10]
print(f"Longest tokens:     {longest}")
print(f"Example:            {tokenizer.encode('First Citizen:' + chr(10) + 'Before we proceed').tokens}")
print(f"Unseen characters:  {tokenizer.encode('café ☃').tokens}")


Vocabulary size     500 (1 special + 65 characters + 434 merges)
Train tokens        407,182 (2.19 characters per token, 0 [UNK])
Val   tokens        105,963 (2.11 characters per token, 0 [UNK])

Longest tokens:     ['GLOUCESTER:', '▁RICHARD', 'GLOUCES', '▁father', '▁should', 'ICHARD', '▁again', '▁blood', '▁death', '▁heart']
Example:            ['First', '▁C', 'it', 'i', 'z', 'en', ':', '\n', 'B', 'e', 'fore', '▁we', '▁pro', 'ce', 'ed']
Unseen characters:  ['c', 'a', 'f', '[UNK]', '▁', '[UNK]']


**Interpretation.**

- The vocabulary is exactly 500 entries: `[UNK]`, the 65 training characters, and 434 learned merges.
- Encoding and decoding reproduce both splits exactly, and neither split contains `[UNK]`.
- Training text averages about 2.2 characters per token and validation text slightly fewer. The merges were learned from training text, so they fit it a little better.
- The longest merges are frequent whole words (`▁father`, `▁blood`) and speaker names (`GLOUCESTER:`). Rare words split into short pieces; for example, `Citizen` becomes `▁C it i z en`.
- The unseen characters in `café ☃` become `[UNK]`, which is how the tokenizer handles symbols missing from the training text.


### 1.3 Token windows

Each split is encoded separately and cut into windows of `N + 1 = 65` tokens. For a window `z`, the input is `z[0:N]` and the target is `z[1:N+1]`, so the target at position `i` is the token that follows input position `i`. The same `N = 64` is used for every configuration.

- **Training windows** start at random offsets, so they may overlap and every position can appear as a target.
- **Validation windows** start at `0, N, 2N, …`; consecutive windows share one boundary token, so each target position is scored exactly once. The final short window is **dropped**, so the loss needs no padding mask.


In [5]:
CONTEXT_LEN = 64  # N

train_ids = torch.tensor(train_encoding.ids, dtype=torch.long)
val_ids = torch.tensor(val_encoding.ids, dtype=torch.long)


def get_train_batch(batch_size, generator):
    """Sample random, possibly overlapping training windows: inputs and targets of shape (B, N)."""
    starts = torch.randint(0, len(train_ids) - CONTEXT_LEN, (batch_size,), generator=generator)
    windows = torch.stack([train_ids[s:s + CONTEXT_LEN + 1] for s in starts.tolist()])
    return windows[:, :-1], windows[:, 1:]


num_val_windows = (len(val_ids) - 1) // CONTEXT_LEN
val_windows = torch.stack([val_ids[i * CONTEXT_LEN:i * CONTEXT_LEN + CONTEXT_LEN + 1] for i in range(num_val_windows)])
val_x, val_y = val_windows[:, :-1], val_windows[:, 1:]
num_val_targets = val_y.numel()

# Each validation target is the next unscored token: positions 1 .. num_val_targets, each once.
assert torch.equal(val_y.reshape(-1), val_ids[1:num_val_targets + 1])
assert torch.equal(val_x[:, 1:], val_y[:, :-1])

print(f"Train tokens                   {len(train_ids):,}")
print(f"Val tokens                     {len(val_ids):,}")
print(f"Validation windows             {num_val_windows:,} x {CONTEXT_LEN} targets")
print(f"Evaluated validation targets   {num_val_targets:,}")
print(f"Dropped (final short window)   {len(val_ids) - 1 - num_val_targets} targets")

x, y = get_train_batch(4, torch.Generator().manual_seed(SEED))
print(f"\nTraining batch shapes: inputs {tuple(x.shape)}, targets {tuple(y.shape)}")
print("First 8 positions of one training window:")
for position in range(8):
    print(f"  input {tokenizer.id_to_token(x[0, position].item())!r:>12}  ->  target {tokenizer.id_to_token(y[0, position].item())!r}")


Train tokens                   407,182
Val tokens                     105,963
Validation windows             1,655 x 64 targets
Evaluated validation targets   105,920
Dropped (final short window)   42 targets

Training batch shapes: inputs (4, 64), targets (4, 64)
First 8 positions of one training window:
  input         'ur'  ->  target 'n'
  input          'n'  ->  target ','
  input          ','  ->  target '▁good'
  input      '▁good'  ->  target '▁C'
  input         '▁C'  ->  target 'at'
  input         'at'  ->  target 'es'
  input         'es'  ->  target 'b'
  input          'b'  ->  target 'y,'


**Interpretation.**

- Training text yields 407,182 tokens. Validation yields 105,963 tokens, cut into 1,655 windows of 64 targets each.
- **105,920 validation targets** are scored. The final short window would have held 42 targets and is dropped; the first validation token is never a target because it has no preceding context.
- The assertions confirm that each validation target is scored once, and that each target row is its input row shifted one token to the left.
- The printed pairs show the shift directly: each target is the next input token.


## 2. Tiny decoder-only Transformer

The model is built from basic layers only (`nn.Embedding`, `nn.Linear`, `nn.RMSNorm`); attention is written with tensor operations. It is checked for causality, then trained in three configurations that differ only in learning rate.


### 2.1 Model

Shapes use `B` = batch size, `T` = sequence length (at most `N = 64`), `D = 128` hidden size, `H = 4` heads of size `D / H = 32`, and `V = 500` vocabulary entries.

- **Input:** token embeddings `(B, T, D)` plus fixed sinusoidal positional encodings `(T, D)`. Attention itself ignores order, so the encodings are the only source of position information.
- **Causal self-attention:** linear projections give queries, keys, and values, reshaped to `(B, H, T, 32)`. Scores `Q Kᵀ / √32` have shape `(B, H, T, T)` with query rows and key columns. Entries with key `j > i` are set to `-inf` **before** the softmax over keys, so their probabilities are exactly zero. The heads are concatenated back to `(B, T, D)` and passed through an output projection.
- **Block (×2):** pre-norm residual layout, `x + Attention(RMSNorm(x))` followed by `x + FFN(RMSNorm(x))`; the feed-forward network is `Linear(128 → 512) → GELU → Linear(512 → 128)`.
- **Output:** a final RMSNorm and a linear projection to raw logits `(B, T, V)`.
- **Dropout:** 0.1 on the embeddings, on the attention probabilities, and on each residual branch; it is inactive in `eval()` mode.

Each attention layer keeps its probabilities after the softmax and before dropout, so they can be plotted in Step 3.


In [6]:
import copy
import json
import math
import time

import torch.nn as nn
import torch.nn.functional as F

D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
FFN_DIM = 4 * D_MODEL
DROPOUT = 0.1


def sinusoidal_encoding(length, dim):
    """Fixed positional encodings (length, dim): sine on even features, cosine on odd features."""
    position = torch.arange(length).unsqueeze(1)
    frequency = torch.exp(-math.log(10000.0) * torch.arange(0, dim, 2) / dim)
    encoding = torch.zeros(length, dim)
    encoding[:, 0::2] = torch.sin(position * frequency)
    encoding[:, 1::2] = torch.cos(position * frequency)
    return encoding


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(context_len, context_len, dtype=torch.bool)))
        self.attn_probs = None  # (B, H, T, T) after softmax, before dropout

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, head_dim)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.split_heads(self.query(x)), self.split_heads(self.key(x)), self.split_heads(self.value(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)  # (B, H, T, T): query rows, key columns
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        probs = scores.softmax(dim=-1)
        self.attn_probs = probs.detach()
        context = self.attn_dropout(probs) @ v  # (B, H, T, head_dim)
        context = context.transpose(1, 2).reshape(B, T, D)  # concatenate heads
        return self.out(context)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, context_len, dropout):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context_len, dropout)
        self.ffn_norm = nn.RMSNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.attn_norm(x)))
        x = x + self.dropout(self.ffn(self.ffn_norm(x)))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, context_len, d_model, n_heads, n_layers, ffn_dim, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer("positional_encoding", sinusoidal_encoding(context_len, d_model))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, ffn_dim, context_len, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        """Token IDs (B, T) -> raw logits (B, T, vocab_size)."""
        x = self.dropout(self.token_embedding(idx) + self.positional_encoding[:idx.shape[1]])
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

    def attention_maps(self):
        """Attention probabilities of the last forward pass, one (B, H, T, T) tensor per layer."""
        return [block.attn.attn_probs for block in self.blocks]


def build_model():
    return TinyTransformer(vocab_size, CONTEXT_LEN, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, DROPOUT)


In [7]:
set_seed(SEED)
model = build_model().to(DEVICE)
num_params = sum(p.numel() for p in model.parameters())
parts = {
    "Token embedding": model.token_embedding,
    "Transformer blocks": model.blocks,
    "Final RMSNorm + output projection": nn.ModuleList([model.final_norm, model.lm_head]),
}
print(f"Parameters  {num_params:,}")
for name, part in parts.items():
    print(f"  {name:<36}{sum(p.numel() for p in part.parameters()):>9,}")

x, y = get_train_batch(4, torch.Generator().manual_seed(SEED))
x, y = x.to(DEVICE), y.to(DEVICE)
model.eval()
with torch.no_grad():
    logits = model(x)
    initial_loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1)).item()
print(f"\nInput {tuple(x.shape)} -> logits {tuple(logits.shape)}")
print(f"Attention per layer {tuple(model.attention_maps()[0].shape)}")
print(f"Loss before training {initial_loss:.3f} (uniform guess: ln {vocab_size} = {math.log(vocab_size):.3f})")


Parameters  524,660
  Token embedding                        64,000
  Transformer blocks                    396,032
  Final RMSNorm + output projection      64,628

Input (4, 64) -> logits (4, 64, 500)
Attention per layer (4, 4, 64, 64)
Loss before training 6.352 (uniform guess: ln 500 = 6.215)


**Interpretation.**

- The model has 524,660 parameters. The two blocks hold 396,032 of them; the embedding and the output projection hold about 64,000 each.
- The shapes match the design: inputs `(4, 64)` produce logits `(4, 64, 500)`, and each layer stores attention probabilities of shape `(4, 4, 64, 64)` (batch, head, query, key).
- Before training, the loss is close to `ln 500 ≈ 6.21`, the loss of a uniform guess over the vocabulary. The untrained model therefore starts with no preference among tokens, as expected.


### 2.2 Causality checks

Both checks run in `eval()` mode, so dropout cannot hide or create differences:

1. **Future attention is zero:** in every layer and head, all probabilities above the diagonal (key `j > i`) must be exactly `0`, and each row must sum to `1`.
2. **Future tokens cannot change earlier outputs:** replace the second half of a validation sequence with random tokens; logits at earlier positions must not change, while later logits should.

As a separate correctness check, the first layer's attention output is compared with PyTorch's `F.scaled_dot_product_attention` using the same weights, inputs, and causal mask, with dropout off. The reference function is used only for this comparison, not in the model.


In [8]:
def causality_checks(model):
    model.eval()
    sequence = val_x[:1].to(DEVICE)  # first validation window, (1, N)
    split = CONTEXT_LEN // 2
    changed = sequence.clone()
    changed[:, split:] = torch.randint(0, vocab_size, (1, CONTEXT_LEN - split), generator=torch.Generator().manual_seed(SEED)).to(DEVICE)

    with torch.no_grad():
        block = model.blocks[0]
        hidden = block.attn_norm(model.token_embedding(sequence) + model.positional_encoding[:CONTEXT_LEN])
        ours = block.attn(hidden)
        attn = block.attn
        q, k, v = attn.split_heads(attn.query(hidden)), attn.split_heads(attn.key(hidden)), attn.split_heads(attn.value(hidden))
        reference = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        reference = attn.out(reference.transpose(1, 2).reshape(hidden.shape))
        reference_diff = (ours - reference).abs().max().item()

        changed_logits = model(changed)
        logits = model(sequence)  # last forward, so attention_maps() holds every layer's maps for `sequence`
        maps = model.attention_maps()
        future = torch.triu(torch.ones(CONTEXT_LEN, CONTEXT_LEN, dtype=torch.bool, device=DEVICE), diagonal=1)
        max_future_weight = max(m[..., future].abs().max().item() for m in maps)
        max_row_sum_error = max((m.sum(dim=-1) - 1).abs().max().item() for m in maps)
        earlier_change = (logits[:, :split] - changed_logits[:, :split]).abs().max().item()
        later_change = (logits[:, split:] - changed_logits[:, split:]).abs().max().item()

    print(f"Max attention weight on future keys        {max_future_weight:.1e}")
    print(f"Max |row sum - 1|                          {max_row_sum_error:.1e}")
    print(f"Max logit change, positions 0-{split - 1} (unchanged) {earlier_change:.1e}")
    print(f"Max logit change, positions {split}-{CONTEXT_LEN - 1} (changed)  {later_change:.3f}")
    print(f"Max difference from reference attention    {reference_diff:.1e}")
    assert max_future_weight == 0.0
    assert max_row_sum_error < 1e-5
    assert earlier_change < 1e-5 and later_change > 1e-3
    assert reference_diff < 1e-5


causality_checks(model)


Max attention weight on future keys        0.0e+00
Max |row sum - 1|                          2.4e-07
Max logit change, positions 0-31 (unchanged) 0.0e+00
Max logit change, positions 32-63 (changed)  3.025
Max difference from reference attention    1.5e-07


**Interpretation.**

- Every attention weight on a future key is exactly `0`: the `-inf` scores become zero probabilities after the softmax, and each row still sums to `1`.
- Replacing tokens 32–63 leaves the logits at positions 0–31 unchanged (differences at floating-point noise level at most), while later logits change. Each prediction therefore depends only on its own and earlier tokens.
- The hand-written attention matches PyTorch's reference implementation to floating-point precision, so the projections, scaling, masking, softmax, and head concatenation are computed correctly.


### 2.3 Training setup

- **Loss:** `cross_entropy` on raw logits and the shifted target IDs. Logits `(B, N, V)` are flattened to `(B·N, V)` and targets `(B, N)` to `(B·N)`; no softmax is applied before the loss, because `cross_entropy` applies log-softmax itself.
- **Optimizer:** AdamW (weight decay 0.01), batch size 64, constant learning rate, gradient-norm clipping at 1.0, and a fixed budget of **5,000 steps** per configuration.
- **Controlled change: learning rate.** The baseline uses `1e-3`; the two other configurations use `3e-4` and `3e-3`. The split, tokenizer, context length, model size, seed, batch order, and step budget are identical, so differences between runs come from the learning rate alone.
- **Validation:** every 100 steps (and before training) the model is scored on all 105,920 validation targets in `eval()` mode under `torch.no_grad()`; validation loss is total negative log-likelihood divided by the number of targets.
- **Checkpoint selection:** each configuration keeps the weights from its lowest-validation-loss evaluation. The configuration with the lowest of these is the selected checkpoint, saved to `artifacts/` with the tokenizer.
- **Training time** counts only the optimization steps; validation passes are timed separately.


In [9]:
BATCH_SIZE = 64
MAX_STEPS = 5000
EVAL_EVERY = 100
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
CONFIGS = {"lr 3e-4": 3e-4, "lr 1e-3 (baseline)": 1e-3, "lr 3e-3": 3e-3}

val_x_device, val_y_device = val_x.to(DEVICE), val_y.to(DEVICE)


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


@torch.no_grad()
def evaluate(model, batch_size=256):
    """Validation cross-entropy: total NLL over all evaluated targets / number of targets."""
    model.eval()
    total_nll = 0.0
    for i in range(0, len(val_x_device), batch_size):
        logits = model(val_x_device[i:i + batch_size])
        targets = val_y_device[i:i + batch_size]
        total_nll += F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1), reduction="sum").item()
    return total_nll / num_val_targets


def train_config(learning_rate):
    set_seed(SEED)  # identical initial weights and dropout masks for every configuration
    model = build_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    batch_generator = torch.Generator().manual_seed(SEED)  # identical batch order for every configuration

    eval_start = time.perf_counter()
    history = {"train_step": [], "train_loss": [], "val_step": [0], "val_loss": [evaluate(model)]}
    eval_seconds = time.perf_counter() - eval_start
    best = {"val_loss": history["val_loss"][0], "step": 0, "state_dict": copy.deepcopy(model.state_dict())}
    sync()
    start = time.perf_counter()
    for step in range(1, MAX_STEPS + 1):
        model.train()
        x, y = get_train_batch(BATCH_SIZE, batch_generator)
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        history["train_step"].append(step)
        history["train_loss"].append(loss.item())

        if step % EVAL_EVERY == 0:
            eval_start = time.perf_counter()
            val_loss = evaluate(model)
            eval_seconds += time.perf_counter() - eval_start
            history["val_step"].append(step)
            history["val_loss"].append(val_loss)
            if val_loss < best["val_loss"]:
                best = {"val_loss": val_loss, "step": step, "state_dict": copy.deepcopy(model.state_dict())}
    sync()
    total_seconds = time.perf_counter() - start
    return {
        "learning_rate": learning_rate,
        "history": history,
        "best": best,
        "train_seconds": total_seconds - eval_seconds,
        "eval_seconds": eval_seconds,
    }


### 2.4 Train the configurations


In [10]:
results = {}
for name, learning_rate in CONFIGS.items():
    results[name] = train_config(learning_rate)
    result = results[name]
    print(
        f"{name:<20} best val loss {result['best']['val_loss']:.4f} at step {result['best']['step']:>4}"
        f" | final val loss {result['history']['val_loss'][-1]:.4f}"
        f" | training {result['train_seconds']:.0f} s, validation {result['eval_seconds']:.0f} s"
    )


lr 3e-4              best val loss 3.4761 at step 4800 | final val loss 3.4790 | training 71 s, validation 6 s


lr 1e-3 (baseline)   best val loss 3.3978 at step 4300 | final val loss 3.4058 | training 71 s, validation 6 s


lr 3e-3              best val loss 3.3507 at step 4700 | final val loss 3.3659 | training 80 s, validation 6 s


**Interpretation.**

- All three runs start from the same weights and see the same batches, so the learning rate is the only difference between them.
- Within the 5,000-step budget, a higher learning rate gives a lower best validation loss: `3e-4` reaches about 3.48 and is still improving at the last step, the baseline `1e-3` about 3.40, and `3e-3` about 3.35.
- At `3e-3`, validation loss flattens after roughly 3,000 steps while training loss keeps falling, the first sign of overfitting. Step 3 examines this with the full loss curves.
- Training time per configuration is similar because the model and budget are identical; small differences come from the hardware, not the learning rate.


### 2.5 Selected checkpoint and run record

The checkpoint with the lowest validation cross-entropy across all configurations is saved with its configuration, and the causality checks are repeated on the trained weights.


In [11]:
selected_name = min(results, key=lambda name: results[name]["best"]["val_loss"])
selected = results[selected_name]
selected_model = build_model().to(DEVICE)
selected_model.load_state_dict(selected["best"]["state_dict"])
assert abs(evaluate(selected_model) - selected["best"]["val_loss"]) < 1e-4

torch.save(
    {
        "config": selected_name,
        "learning_rate": selected["learning_rate"],
        "step": selected["best"]["step"],
        "val_loss": selected["best"]["val_loss"],
        "model_args": dict(vocab_size=vocab_size, context_len=CONTEXT_LEN, d_model=D_MODEL, n_heads=N_HEADS,
                           n_layers=N_LAYERS, ffn_dim=FFN_DIM, dropout=DROPOUT),
        "state_dict": selected["best"]["state_dict"],
    },
    ARTIFACTS_DIR / "selected_checkpoint.pt",
)
(ARTIFACTS_DIR / "training_histories.json").write_text(
    json.dumps({name: result["history"] for name, result in results.items()})
)

hardware = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else cpu_model()
print(f"Selected checkpoint  {selected_name}, step {selected['best']['step']}, val loss {selected['best']['val_loss']:.4f}")
print(f"Optimizer            AdamW (weight decay {WEIGHT_DECAY}), gradient clipping {GRAD_CLIP}")
print(f"Batch size           {BATCH_SIZE} windows x {CONTEXT_LEN} tokens")
print(f"Steps                {MAX_STEPS:,} per configuration (validation every {EVAL_EVERY})")
print(f"Parameters           {num_params:,}")
print(f"Hardware             {hardware} ({DEVICE.type})")
print(f"Packages             PyTorch {torch.__version__}, tokenizers {tokenizers.__version__}, Python {platform.python_version()}")
print(f"Seed                 {SEED}")
print()
causality_checks(selected_model)


Selected checkpoint  lr 3e-3, step 4700, val loss 3.3507
Optimizer            AdamW (weight decay 0.01), gradient clipping 1.0
Batch size           64 windows x 64 tokens
Steps                5,000 per configuration (validation every 100)
Parameters           524,660
Hardware             Apple M5 (mps)
Packages             PyTorch 2.8.0, tokenizers 0.22.2, Python 3.9.6
Seed                 42



Max attention weight on future keys        0.0e+00
Max |row sum - 1|                          2.4e-07
Max logit change, positions 0-31 (unchanged) 0.0e+00
Max logit change, positions 32-63 (changed)  20.254
Max difference from reference attention    7.2e-07


**Interpretation.**

- The selected checkpoint is the `3e-3` configuration at its lowest-validation-loss step; re-evaluating the saved weights reproduces the recorded loss.
- The causality checks pass on the trained weights as well, so causality is enforced by the mask and does not depend on the weights.
- The printed record lists everything the handout requires for each configuration (optimizer, batch size, learning rate, steps, parameter count, hardware, package versions, seed); training times are in the table above.
